<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/02-Basic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Packages and Setup Variables


In [ ]:
!pip install -q openai==2.8.1 google-genai==1.51.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.5/260.5 kB 20.9 MB/s eta 0:00:00


In [ ]:
import os

# Set the "OPENAI_API_KEY" and "GOOGLE_API_KEY" in the Python environment. Will be used by OpenAI client later.

os.environ["OPENAI_API_KEY"] = "<YOUR_OPENAI_API_KEY>"
os.environ["GOOGLE_API_KEY"] =  "<YOUR_GOOGLE_API_KEY>"

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] =  userdata.get('Google_api_key')

In [ ]:
# False: Generate the embedding for the dataset. (Associated cost with using OpenAI endpoint)
# True: Load the dataset that already has the embedding vectors.
load_embedding = True

# Load Dataset


## Download Dataset (JSON)


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv

--2025-12-17 16:53:45--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 173646 (170K) [text/plain]
Saving to: ‘mini-llama-articles.csv.1’

mini-llama-articles 100%[===================>] 169.58K  --.-KB/s    in 0.03s   

2025-12-17 16:53:45 (6.28 MB/s) - ‘mini-llama-articles.csv.1’ saved [173646/173646]

--2025-12-17 16:53:45--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HT

## Read File


In [ ]:
# Split the input text into chunks of specified size. (chunks by chars now, in actual work consider chunks by tokens)
def split_into_chunks(text, chunk_size=1024):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i : i + chunk_size])

    return chunks

In [ ]:
import csv

chunks = []

# Load the file as a CSV
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        chunks.extend(split_into_chunks(row[1]))

In [ ]:
print("number of articles:", idx)
print("number of chunks:", len(chunks))

number of articles: 14
number of chunks: 174


In [ ]:
import pandas as pd

# Convert the JSON list to a Pandas Dataframe
df = pd.DataFrame(chunks, columns=["chunk"])

df.keys()

# index	chunk visualisation:
# 0	"first chunk text..."
# 1	"second chunk text..."
# 2	"third chunk text..."
# 3	"..."
# 4	"..."

Index(['chunk'], dtype='object')

# Generate Embedding


An embedding is a vector (a long list of numbers) that represents the semantic meaning of some data — usually text.

In [ ]:
from openai import OpenAI
client = OpenAI()

# Defining a function that converts a text to embedding vector using OpenAI's Ada model.
def get_embedding(text):
    try:
        # Remove newlines
        text = text.replace("\n", " ")
        res = client.embeddings.create(input=[text], model="text-embedding-3-small")

        return res.data[0].embedding

    except:
        return None

# import google.generativeai as genai
# from google.colab import userdata

# genai.configure(api_key=userdata.get("Google_api_key"))

# def get_embedding(text: str):
#     result = genai.embed_content(
#         model="models/embedding-001",
#         content=text
#     )
#     return result["embedding"]

In [48]:
pip install nbstripout

In [50]:
!nbstripout 02-Basic_RAG.ipynb

Could not strip '02-Basic_RAG.ipynb': file not found


In [ ]:
# from tqdm.notebook import tqdm
from tqdm import tqdm
import numpy as np
import ast

# Generate embedding
if not load_embedding:
    print("Generating embeddings...")
    embeddings = []
    for index, row in tqdm(df.iterrows()):
        # df.at[index, 'embedding'] = get_embedding( row['chunk'] )
        embeddings.append(get_embedding(row["chunk"]))

    embeddings_values = pd.Series(embeddings)
    df.insert(loc=1, column="embedding", value=embeddings_values)

# Or, load the embedding from the file.
else:
    print("Loaded the embedding file.")
    # Load the file as a CSV
    df = pd.read_csv("mini-llama-articles-with_embeddings.csv")

    # Convert embedding column to an array

    # df["embedding"] = df["embedding"].apply(lambda x: np.array(eval(x)), 0)

    df["embedding"] = (
        df["embedding"]
        .astype(object)
        .apply(lambda x: np.array(ast.literal_eval(x)))
    )

Generating embeddings...


0it [00:00, ?it/s]

In [ ]:
df.to_csv('mini-llama-articles-with_embeddings.csv')

# User Question


In [ ]:
# Define the user question, and convert it to embedding.
QUESTION = "How many parameters LLaMA2 model has?"
QUESTION_emb = get_embedding(QUESTION)

len(QUESTION_emb)

1536

# Test Cosine Similarity


Calculating the similarity of embedding representations can help us to find pieces of text that are close to each other. In the following sample you see how the Cosine Similarity metric can identify which sentence could be a possible answer for the given user question. Obviously, the unrelated answer will score lower.


In [ ]:
BAD_SOURCE_emb = get_embedding("The sky is blue.")
GOOD_SOURCE_emb = get_embedding("LLaMA2 model has a total of 2B parameters.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# A sample that how a good piece of text can achieve high similarity score compared
# to a completely unrelated text.
print("> Bad Response Score:", cosine_similarity([QUESTION_emb], [BAD_SOURCE_emb]))
print("> Good Response Score:", cosine_similarity([QUESTION_emb], [GOOD_SOURCE_emb]))

> Bad Response Score: [[0.02578592]]
> Good Response Score: [[0.8315133]]


# Calculate Cosine Similarities


In [ ]:
# The similarity between the questions and each part of the essay.
cosine_similarities = cosine_similarity([QUESTION_emb], df["embedding"].tolist())

print(cosine_similarities)

[[0.4676531  0.4690562  0.25974209 0.29394356 0.3195583  0.40153824
  0.41498478 0.45270485 0.45926732 0.12609017 0.11750975 0.01343921
  0.22601635 0.21421833 0.10134789 0.33065192 0.10740536 0.34678967
  0.16306555 0.08731621 0.34819773 0.22842265 0.19201108 0.2646903
  0.24924728 0.34828784 0.24829788 0.32759326 0.41444925 0.412942
  0.45763978 0.38341387 0.46859561 0.35635197 0.3539408  0.30140312
  0.29937216 0.29253776 0.40024316 0.46475934 0.39467758 0.4103955
  0.4471235  0.43178344 0.35902344 0.33980702 0.51354144 0.20928826
  0.40201365 0.32874487 0.42832614 0.48175429 0.45030102 0.34253433
  0.32091127 0.42590468 0.24627486 0.18087341 0.23644698 0.3426414
  0.34366738 0.20465058 0.19752012 0.22434051 0.21113429 0.42299409
  0.26378762 0.3045309  0.33611993 0.3829711  0.23529294 0.24351179
  0.37086692 0.28031518 0.49051146 0.53037973 0.37814241 0.43761385
  0.37744446 0.39258483 0.30060333 0.41717781 0.4673489  0.45420816
  0.35134751 0.21203511 0.42618926 0.31603956 0.44057

In [ ]:
import numpy as np

number_of_chunks_to_retrieve = 3

# Sort the scores
highest_index = np.argmax(cosine_similarities)

# Pick the N highest scored chunks
indices = np.argsort(cosine_similarities[0])[::-1][:number_of_chunks_to_retrieve]
print(indices)

[114  75  89]


In [ ]:
# Look at the highest scored retrieved pieces of text
for idx, item in enumerate(df.chunk[indices]):
    print(f"> Chunk {idx+1}")
    print(item)
    print("----")

> Chunk 1
by Meta that ventures into both the AI and academic spaces. The model aims to help researchers, scientists, and engineers advance their work in exploring AI applications. It will be released under a non-commercial license to prevent misuse, and access will be granted to academic researchers, individuals, and organizations affiliated with the government, civil society, academia, and industry research facilities on a selective case-by-case basis. The sharing of codes and weights allows other researchers to test new approaches in LLMs. The LLaMA models have a range of 7 billion to 65 billion parameters. LLaMA-65B can be compared to DeepMind's Chinchilla and Google's PaLM. Publicly available unlabeled data was used to train these models, and training smaller foundational models require less computing power and resources. LLaMA 65B and 33B have been trained on 1.4 trillion tokens in 20 different languages, and according to the Facebook Artificial Intelligence Research (FAIR) team,

# Augment the Prompt


In [ ]:
from google import genai
from google.genai import types
from google.genai.types import HttpOptions

# Initialize client
client = genai.Client(api_key=userdata.get('Google_api_key'))

# Use the Gemini API to answer the questions based on the retrieved pieces of text.
try:
    # System instructions for the AI assistant
    system_instruction = (
        "You are an assistant and expert in answering questions from a chunks of content. "
        "Only answer AI-related question, else say that you cannot answer this question."
    )

    # Create a user prompt with the user's question
    prompt = (
        "Read the following informations that might contain the context you require to answer the question. You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
        "Please provide an informative and accurate answer to the following question based on the available context. Be concise and take your time. \nQuestion: {}\nAnswer:"
    )

    # Add the retrieved pieces of text to the prompt (top 3 chunks)
    formatted_prompt = prompt.format("".join(df.chunk[indices]), QUESTION)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=formatted_prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0),  # “Don’t overthink it. Just answer.”
            system_instruction=system_instruction,
            temperature=0.0,  # Low temperature → predictable
        )
    )

    res = response.text

except Exception as e:
    print(f"An error occurred: {e}")
# print(formatted_prompt)
# print("=====")
print(res)

I cannot answer this question. The provided text does not contain information about a "LLaMA 4" model or its parameters. It discusses LLaMA (Llama 1) and Llama 2, including their parameter ranges.


## How Augmenting the Prompt can address knowledge cutoff limitations and hallucinations

In [ ]:
# Consider this as a retrieved chunk
# https://ai.meta.com/blog/llama-4-multimodal-intelligence (Summarized Content)

Example_chunk ="""
Meta has unveiled the **Llama 4 herd**, a new generation of open-weight, natively multimodal AI models designed to push the boundaries of performance, efficiency, and accessibility. The release includes **Llama 4 Scout** and **Llama 4 Maverick**, alongside a preview of the massive **Llama 4 Behemoth** teacher model. These models introduce advanced **mixture-of-experts (MoE) architectures**, native text–image integration, and record-breaking context lengths, establishing Llama 4 as a major leap in multimodal AI innovation.
**Llama 4 Scout** is a **17B active parameter model with 16 experts** (109B total parameters) that offers **10 million token context length**, far exceeding Llama 3’s 128K limit. Scout’s architecture leverages **interleaved attention layers (iRoPE)** and inference-time temperature scaling to achieve superior length generalization, enabling complex tasks like multi-document summarization, codebase reasoning, and long-context retrieval. Despite its smaller size, Scout surpasses prior Llama models and competitors such as Gemma 3 and Gemini 2.0 Flash-Lite in performance, all while being deployable on a **single NVIDIA H100 GPU**. Its multimodal training allows strong image grounding, multi-image reasoning, and visual question answering.
**Llama 4 Maverick**, also with **17B active parameters**, uses **128 experts** and totals **400B parameters**, alternating dense and MoE layers for inference efficiency. It rivals or exceeds larger models like **GPT-4o, Gemini 2.0 Flash, and DeepSeek v3** on benchmarks for coding, reasoning, multilinguality, and multimodal tasks. Its efficient design makes it deployable on a single **NVIDIA H100 DGX host**, balancing performance with cost-effectiveness. Maverick was refined through a revamped **post-training pipeline**—lightweight supervised fine-tuning, continuous **online reinforcement learning (RL)** with adaptive difficulty filtering, and direct preference optimization—resulting in superior reasoning, conversational fluency, and multimodal understanding. Maverick’s **chat version achieved an ELO score of 1417 on LMArena**, reflecting best-in-class general assistant capabilities.
At the top of the hierarchy, **Llama 4 Behemoth** serves as the **teacher model**, with **288B active parameters, 16 experts, and nearly 2 trillion total parameters**. It outperforms **GPT-4.5, Claude Sonnet 3.7, and Gemini 2.0 Pro** on STEM benchmarks like MATH-500 and GPQA Diamond. Behemoth’s scale demanded innovative training strategies, including pruning 95% of supervised fine-tuning data, advanced reinforcement learning with dynamically filtered prompts, and an optimized asynchronous MoE-parallelized RL infrastructure. These techniques boosted reasoning, coding, and multilingual capabilities while maintaining instruction-following reliability.
All Llama 4 models are **natively multimodal**, trained with large-scale text, image, and video data, and leverage **Meta’s MetaP technique** for reliable hyperparameter scaling. They support **200 languages**, with 10x more multilingual tokens than Llama 3, and employ **FP8 precision** for efficient training across trillions of tokens. Safety remains a priority: Meta integrates **Llama Guard**, **Prompt Guard**, and **CyberSecEval** for content protection, while **Generative Offensive Agent Testing (GOAT)** automates adversarial red-teaming. Llama 4 also significantly reduces **political and social bias**, refusing fewer prompts and responding more neutrally than Llama 3.
In sum, **Llama 4 Scout, Maverick, and Behemoth** represent a major leap in open AI research: compact yet powerful models with unmatched context length, multimodal fluency, reasoning power, and efficiency. By making Scout and Maverick openly available on **llama.com and Hugging Face**, Meta empowers developers, enterprises, and researchers worldwide to build the next generation of AI-driven experiences."""

In [ ]:
QUESTION = "How many parameters does LLaMA 4 model have?"

system_instruction = (
    "You are an assistant and expert in answering questions from a chunks of content. "
    "Only answer AI-related question, else say that you cannot answer this question."
)

# Combining the context with the user's question
prompt = (
    "Read the following informations that might contain the context you require to answer the question. You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
    "Please provide an informative and accurate answer to the following question based on the available context. Be concise and take your time. \nQuestion: {}\nAnswer:"
)

formatted_prompt = prompt.format(Example_chunk, QUESTION)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=formatted_prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction=system_instruction,
        temperature=0.0,
    )
)

res = response.text
print(formatted_prompt)
print("=====")
print(res)

Read the following informations that might contain the context you require to answer the question. You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:

<START_OF_CONTEXT>

Meta has unveiled the **Llama 4 herd**, a new generation of open-weight, natively multimodal AI models designed to push the boundaries of performance, efficiency, and accessibility. The release includes **Llama 4 Scout** and **Llama 4 Maverick**, alongside a preview of the massive **Llama 4 Behemoth** teacher model. These models introduce advanced **mixture-of-experts (MoE) architectures**, native text–image integration, and record-breaking context lengths, establishing Llama 4 as a major leap in multimodal AI innovation.
**Llama 4 Scout** is a **17B active parameter model with 16 experts** (109B total parameters) that offers **10 million token context length**, far exceeding Llama 3’s 128K limit. Scout’s architecture leverages **interleave

# Without Augmentation


Test the Gemini API to answer the same question without the addition of retrieved documents. Basically, the LLM will use its knowledge to answer the question.


In [ ]:
# Test without retrieved documents
QUESTION = "How many parameters does LLaMA 4 model have?"

# System instructions
system_instruction = "You are an assistant and expert in answering questions."

# Simple prompt without context
prompt = "Be concise and take your time to answer the following question. \nQuestion: {}\nAnswer:"
formatted_prompt = prompt.format(QUESTION)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=formatted_prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction=system_instruction,
        temperature=0.0,
    )
)

res = response.text

print(formatted_prompt)
print("=====")
print(res)


Be concise and take your time to answer the following question. 
Question: How many parameters does LLaMA 4 model have?
Answer:
=====
LLaMA 4 has not been released. The latest model is LLaMA 3.
